In [ ]:
import pandas as pd
import re
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats.mstats import winsorize
from sklearn.preprocessing import StandardScaler



Merge Data


In [ ]:
#merge data
kagglepath =  "../../dataset/kgdataset.csv"
surveypath = "../../dataset/survey_data.xlsx"
df = pd.read_csv(kagglepath)
df2 = pd.read_excel(surveypath)
df2.drop(df2.columns[0],axis=1, inplace=True)
df2.columns = df.columns
combined = pd.concat([df, df2], ignore_index=True)
combined


Cleandata

In [ ]:
dfclean = pd.DataFrame(combined)
dfclean["Feel Rested"] = dfclean["Feel Rested"].replace("ไม่สดชื่น", "No").replace("สดชื่น", "Yes")
dfclean["Use Before Sleep"] = dfclean["Use Before Sleep"].replace("ไม่ใช่", "No").replace("ใช่", "Yes")
dfclean["Anxiety/Low Mood"] = dfclean["Anxiety/Low Mood"].replace("ไม่หงุดหงิด/ไม่วิตกกังวล", "No").replace("หงุดหงิด/วิตกกังวล", "Yes")
dfclean["Wellness Apps"] = dfclean["Wellness Apps"].replace("ไม่ใช่", "No").replace("ใช่", "Yes")
dfclean["Sleep Quality"] = dfclean["Sleep Quality"].replace("ไม่ดี", "Bad").replace("ดี", "Good")
dfclean["Screen Time Affects Sleep?"] = dfclean["Screen Time Affects Sleep?"].replace("ไม่แน่ใจ", "Not Sure").replace("ใช่", "Yes").replace("ไม่ใช่", "No")
#check for null values and duplicates
duplicates = dfclean[dfclean.duplicated()]

dfclean = dfclean.drop_duplicates()

#clean age
def extract_number(value):
    if "-" in str(value):
        value = re.sub(r'[^\d\-\.]', '', value)
        values = str(value).split("-")
        return f"{values[0]}-{values[1]}"
    elif isinstance(value, str):
        result = pd.Series(value).str.extract('(\d+)') 
        return result[0].iloc[0] if not result.empty else None

    else:
        return str(value)      
dfclean["Age"] = dfclean["Age"].apply(extract_number)

#clean sleep hours by mean
def clean_range_with_mean(value):
    if isinstance(value, str):
        if '-' in value:
            try:
                low, high = map(float, value .split('-'))
                return (low + high) / 2
            except ValueError:
                return value
        else:
            try:
                return float(value)
            except ValueError:
                return value
    else:
        return value
dfclean["Sleep Hours"] = dfclean["Sleep Hours"].apply(extract_number)
dfclean["Sleep Hours"] = dfclean["Sleep Hours"].apply(clean_range_with_mean)

#Clean Daily Screen Time
dfclean["Daily Screen Time"] = dfclean["Daily Screen Time"].apply(extract_number)
dfclean["Daily Screen Time"] = dfclean["Daily Screen Time"].apply(clean_range_with_mean)

#handle outliers by winsorization
dfclean['Daily Screen Time'] = winsorize(dfclean['Daily Screen Time'],(0.1, 0.1))
dfclean['Age'] = winsorize(dfclean['Age'],(0.1, 0.1))



Tranform

In [ ]:
binary_columns = [ "Use Before Sleep", "Anxiety/Low Mood", "Wellness Apps"]
for i in range(len(binary_columns)):
    dfclean[binary_columns[i]] = dfclean[binary_columns[i]].map({"Yes": 1, "No": 0})
dfclean["Feel Rested"] = dfclean["Feel Rested"].map({"Yes": 2, "Sometimes": 1, "No": 0})
dfclean["Sleep Quality"] = dfclean["Sleep Quality"].map({"Good": 1, "Bad": 0})
dfclean["Screen Time Affects Sleep?"] = dfclean["Screen Time Affects Sleep?"].map({"Yes": 1, "No": 0, "Not Sure": 2})

Standard Scaler

In [ ]:
scaler = StandardScaler()
numeric_columns = ["Age", "Sleep Hours", "Daily Screen Time","Stress Level"]
dfclean[numeric_columns] = scaler.fit_transform(dfclean[numeric_columns])
dfclean